In [ ]:
!git clone https://github.com/wmcnally/golfdb.git


In [ ]:
pip install torch torchvision opencv-python numpy pandas tqdm


In [ ]:
!cp /kaggle/input/datasets/limmaximus/1119-video/1119.mp4 /kaggle/working/golfdb


In [ ]:
import os
os.makedirs('/kaggle/working/golfdb/models', exist_ok=True)

!cp /kaggle/input/swingnet-1800-pth-tar/pytorch/default/1/swingnet_1800.pth.tar /kaggle/working/golfdb/models/

In [ ]:
# Read the current model.py file
with open('/kaggle/working/golfdb/model.py', 'r') as f:
    content = f.read()

# Replace the problematic lines
old_code = """        net = MobileNetV2(width_mult=width_mult)
        state_dict_mobilenet = torch.load('mobilenet_v2.pth.tar')
        if pretrain:
            net.load_state_dict(state_dict_mobilenet)"""

new_code = """        net = MobileNetV2(width_mult=width_mult)
        if pretrain:
            from torchvision.models import mobilenet_v2
            try:
                mobilenet = mobilenet_v2(pretrained=True)
            except:
                from torchvision.models import MobileNet_V2_Weights
                mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
            net.load_state_dict(mobilenet.features.state_dict())"""

content = content.replace(old_code, new_code)

# Write the modified content back
with open('/kaggle/working/golfdb/model.py', 'w') as f:
    f.write(content)

print("model.py has been modified successfully!")

In [ ]:
# Read the test_video.py file
with open('/kaggle/working/golfdb/test_video.py', 'r') as f:
    content = f.read()

# Replace pretrain=True with pretrain=False
content = content.replace('model = EventDetector(pretrain=True,', 
                         'model = EventDetector(pretrain=False,')

# Write the modified content back
with open('/kaggle/working/golfdb/test_video.py', 'w') as f:
    f.write(content)

print("File modified successfully!")
print("pretrain=True changed to pretrain=False")

In [ ]:
# Read the file
with open('/kaggle/working/golfdb/test_video.py', 'r') as f:
    content = f.read()

# Fix the model path - use absolute path
content = content.replace(
    "save_dict = torch.load('models/swingnet_1800.pth.tar')",
    "save_dict = torch.load('/kaggle/working/golfdb/models/swingnet_1800.pth.tar')"
)

# Write back
with open('/kaggle/working/golfdb/test_video.py', 'w') as f:
    f.write(content)

print("Updated model path to absolute path!")

In [ ]:
# Read the file
with open('/kaggle/working/golfdb/test_video.py', 'r') as f:
    content = f.read()

# Fix 1: Change pretrain to False (if not already done)
content = content.replace('pretrain=True', 'pretrain=False')

# Fix 2: Use absolute path for model loading
content = content.replace(
    "save_dict = torch.load('models/swingnet_1800.pth.tar')",
    "save_dict = torch.load('/kaggle/working/golfdb/models/swingnet_1800.pth.tar')"
)

# Fix 3: Add exit() to the except block
old_except = """    except:
        print("Model weights not found. Download model weights and place in 'models' folder. See README for instructions")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')"""

new_except = """    except Exception as e:
        print("Model weights not found. Download model weights and place in 'models' folder. See README for instructions")
        print(f"Error: {e}")
        exit()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')"""

content = content.replace(old_except, new_except)

# Write back
with open('/kaggle/working/golfdb/test_video.py', 'w') as f:
    f.write(content)

print("All fixes applied!")

In [ ]:
import os
os.chdir('/kaggle/working/golfdb')
!python test_video.py -p 1119.mp4

In [ ]:
!cd /kaggle/working/golfdb && python3 test_video.py -p test_video.mp4

In [ ]:
with open('/kaggle/working/golfdb/model.py', 'r') as f:
    content = f.read()

old = """        net = MobileNetV2(width_mult=width_mult)
        state_dict_mobilenet = torch.load('mobilenet_v2.pth.tar')
        if pretrain:
            net.load_state_dict(state_dict_mobilenet)"""

new = """        net = MobileNetV2(width_mult=width_mult)
        if pretrain:
            state_dict_mobilenet = torch.load('mobilenet_v2.pth.tar')
            net.load_state_dict(state_dict_mobilenet)"""

content = content.replace(old, new)

with open('/kaggle/working/golfdb/model.py', 'w') as f:
    f.write(content)

print("Fixed!")

In [ ]:
import subprocess

# First check the input
!ffprobe -v quiet -show_format -show_streams /kaggle/working/golfdb/test_video.mp4 2>/dev/null | grep -E "(nb_frames|r_frame_rate|width|height)"

# Slow down using ffmpeg (duplicate frames)
!ffmpeg -i /kaggle/working/golfdb/pro_swing6.mp4 -filter:v "setpts=4*PTS" -an /kaggle/working/golfdb/pro_swing6_slow.mp4 -y

# Verify output
!ffprobe -v quiet -show_streams /kaggle/working/golfdb/pro_swing6_slow.mp4 2>/dev/null | grep -E "(nb_frames|r_frame_rate)"

In [ ]:
!ffmpeg -i /kaggle/working/golfdb/test_video.mp4 -filter:v "setpts=4*PTS" -an /kaggle/working/golfdb/test_video.mp4 -y